# NBA Dataset Exploration
This notebook explores the NBA games dataset to understand its structure and contents.

# Stage 1: Data Understanding and Baseline
Load data, understand structure, create first target variable.

1. Load data

In [ ]:
import pandas as pd

df = pd.read_csv("../data/nba_games.csv")
df.head()
df.info()
print(df.shape)

2. Create target variable

In [ ]:
# 1 = home win, 0 = away win
df["HOME_WIN"] = (df["homeScore"] > df["awayScore"]).astype(int)

3. Baseline metric

In [ ]:
home_win_rate = df["HOME_WIN"].mean()
print(f"Home Win Rate: {home_win_rate:.2%}")

away_win_rate = 1 - home_win_rate
print(f"Away Win Rate: {away_win_rate:.2%}")

# Stage 2: Team Strength
Turn raw games into team-level performance metrics.

1. Filter modern era

In [ ]:
df_modern = df[df["gameDate"] >= "2000-01-01"].copy()
df_modern = df_modern.sort_values("gameDate")

2. Recreate target

In [ ]:
df_modern["HOME_WIN"] = (df_modern["homeScore"] > df_modern["awayScore"]).astype(int)
df_modern["AWAY_WIN"] = (df_modern["awayScore"] > df_modern["homeScore"]).astype(int)

3. Team strength (home + away)

In [ ]:
home_strength = df_modern.groupby("hometeamName")["HOME_WIN"].mean()
away_strength = df_modern.groupby("awayteamName")["AWAY_WIN"].mean()

4. Combine into one table

In [ ]:
team_strength = pd.DataFrame({
    "home_strength": home_strength,
    "away_strength": away_strength
})

team_strength["overall_strength"] = (
    team_strength["home_strength"] + team_strength["away_strength"]
) / 2

team_strength.head()

# Stage 3: Prediction Logic
Turn team strengths into game predictions.

1. Build game-level dataset

In [ ]:
home_games = df_modern[["hometeamName", "gameDate", "HOME_WIN"]].copy()
home_games.columns = ["team", "date", "win"]

away_games = df_modern[["awayteamName", "gameDate", "AWAY_WIN"]].copy()
away_games.columns = ["team", "date", "win"]

all_games = pd.concat([home_games, away_games])
all_games = all_games.sort_values(["team", "date"])

2. Recent form (weighted)

In [ ]:
def weighted_form(x):
    # weights = [0.05, 0.1, 0.15, 0.25, 0.45]
    weights = [0.02, 0.03, 0.05, 0.07, 0.10, 0.12, 0.15, 0.18, 0.20, 0.28]
    
    return x.rolling(10, min_periods=1).apply(
        lambda g: sum(g * weights[-len(g):]),
        raw=True
    )

all_games["recent_form"] = (
    all_games.groupby("team")["win"]
    .apply(weighted_form)
    .reset_index(level=0, drop=True)
)


3. Build final team table

In [ ]:
recent_form = all_games.groupby("team").tail(1)[["team", "recent_form"]]

team_strength = team_strength.reset_index()

team_strength = team_strength.merge(
    recent_form,
    left_on="index",
    right_on="team",
    how="left"
)

team_strength = team_strength.drop(columns=["team"])

4. Prediction function

In [ ]:
def predict_game(home_team, away_team):

    home_row = team_strength[team_strength["index"] == home_team].iloc[0]
    away_row = team_strength[team_strength["index"] == away_team].iloc[0]

    home_score = (
        0.8 * home_row["overall_strength"] +
        0.5 * home_row["recent_form"]
    )

    away_score = (
        0.8 * away_row["overall_strength"] +
        0.5 * away_row["recent_form"]
    )

    total = home_score + away_score

    home_prob = home_score / total
    away_prob = away_score / total

    winner = home_team if home_prob > away_prob else away_team

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_prob": float(round(home_prob, 3)),
        "away_prob": float(round(away_prob, 3)),
        "predicted_winner": winner
    }

5. Test

In [ ]:
predict_game("Heat", "Bucks")
predict_game("Knicks", "Celtics")

6. Evaluation

In [ ]:
# 6. Evaluation

test_games = df_modern[[
    "hometeamName",
    "awayteamName",
    "homeScore",
    "awayScore"
]].copy()

test_games = test_games.sample(200, random_state=42)

test_games["actual_winner_name"] = test_games.apply(
    lambda row: row["hometeamName"] if row["homeScore"] > row["awayScore"] else row["awayteamName"],
    axis=1
)

test_games["predicted_winner"] = test_games.apply(
    lambda row: predict_game(row["hometeamName"], row["awayteamName"])["predicted_winner"],
    axis=1
)

accuracy = (
    test_games["predicted_winner"] ==
    test_games["actual_winner_name"]
).mean()

print(f"\nModel Accuracy: {accuracy:.2%}")

print("\nSample Predictions:")

sample = test_games[[
    "hometeamName",
    "awayteamName",
    "predicted_winner",
    "actual_winner_name"
]].head(10)

sample.columns = [
    "Home Team",
    "Away Team",
    "Predicted Winner",
    "Actual Winner"
]

display(sample.reset_index(drop=True))

# Stage 4: Machine Learning Model

1. Build training dataset

In [ ]:
ml_df = df_modern[[
    "hometeamName",
    "awayteamName",
    "homeScore",
    "awayScore"
]].copy()

# target
ml_df["HOME_WIN"] = (ml_df["homeScore"] > ml_df["awayScore"]).astype(int)

2. Add features from Stage 2/3 for home and away

In [ ]:
ml_df = ml_df.merge(
    team_strength[["index", "overall_strength", "recent_form"]],
    left_on="hometeamName",
    right_on="index",
    how="left"
).rename(columns={
    "overall_strength": "home_strength",
    "recent_form": "home_form"
}).drop(columns=["index"])

ml_df = ml_df.merge(
    team_strength[["index", "overall_strength", "recent_form"]],
    left_on="awayteamName",
    right_on="index",
    how="left"
).rename(columns={
    "overall_strength": "away_strength",
    "recent_form": "away_form"
}).drop(columns=["index"])

3. Fix NaN issue

In [ ]:
ml_df = ml_df.dropna(subset=[
    "home_strength",
    "away_strength",
    "home_form",
    "away_form"
])

4. Select features

In [ ]:
features = [
    "home_strength",
    "away_strength",
    "home_form",
    "away_form"
]

X = ml_df[features]
y = ml_df["HOME_WIN"]

5. Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

6. Train model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

7. Evaluate

In [ ]:
from sklearn.metrics import accuracy_score

preds = model.predict(X_test)

ml_accuracy = accuracy_score(y_test, preds)
print(f"ML Model Accuracy: {ml_accuracy:.2%}")

8. Replace your predict_game (important)

In [ ]:
def predict_game_ml(home_team, away_team):

    home = team_strength[team_strength["index"] == home_team].iloc[0]
    away = team_strength[team_strength["index"] == away_team].iloc[0]

    input_data = pd.DataFrame([{
        "home_strength": home["overall_strength"],
        "away_strength": away["overall_strength"],
        "home_form": home["recent_form"],
        "away_form": away["recent_form"]
    }])

    prob = model.predict_proba(input_data)[0]

    home_prob = prob[1]
    away_prob = prob[0]

    winner = home_team if home_prob > away_prob else away_team

    print(f"\n🏀 {home_team} vs {away_team}")
    print(f"{home_team}: {home_prob:.1%}")
    print(f"{away_team}: {away_prob:.1%}")
    print(f"Predicted Winner: {winner}")

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_prob": round(float(home_prob), 3),
        "away_prob": round(float(away_prob), 3),
        "predicted_winner": winner
    }

In [ ]:
predict_game_ml("Heat", "Bucks")

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Rule-Based",
        "Logistic Regression"
    ],
    "Accuracy": [
        f"{accuracy:.2%}",
        f"{ml_accuracy:.2%}"
    ]
})

comparison

# Stage 5: Feature Engineering

Improve the model by adding stronger features.

first additional feature is point differential

In [ ]:
df_modern["point_diff"] = df_modern["homeScore"] - df_modern["awayScore"]

In [ ]:
home_pd = df_modern.groupby("hometeamName")["point_diff"].mean()

away_pd = (
    df_modern.groupby("awayteamName")["point_diff"]
    .mean() * -1
)

point_diff = pd.DataFrame({
    "home_point_diff": home_pd,
    "away_point_diff": away_pd
})

point_diff["avg_point_diff"] = (
    point_diff["home_point_diff"] +
    point_diff["away_point_diff"]
) / 2

point_diff.head()

In [ ]:
team_strength = team_strength.merge(
    point_diff[["avg_point_diff"]],
    left_on="index",
    right_index=True,
    how="left"
)
team_strength.head()

next i will be adding rolling point differential

In [ ]:
df_modern["point_diff"] = df_modern["homeScore"] - df_modern["awayScore"]

In [ ]:
home_pd = df_modern[["hometeamName", "gameDate", "point_diff"]].copy()
home_pd.columns = ["team", "date", "point_diff"]

away_pd = df_modern[["awayteamName", "gameDate", "point_diff"]].copy()
away_pd["point_diff"] = -away_pd["point_diff"]
away_pd.columns = ["team", "date", "point_diff"]

all_pd = pd.concat([home_pd, away_pd])
all_pd = all_pd.sort_values(["team", "date"])

In [ ]:
all_pd["rolling_pd"] = (
    all_pd.groupby("team")["point_diff"]
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
rolling_pd_latest = all_pd.groupby("team").tail(1)[["team", "rolling_pd"]]

In [ ]:
team_strength = team_strength.merge(
    rolling_pd_latest,
    left_on="index",
    right_on="team",
    how="left"
).drop(columns=["team"])

In [ ]:
team_strength = team_strength.rename(columns={
    "rolling_pd": "rolling_point_diff"
})

In [ ]:
features = [
    "home_strength",
    "away_strength",
    "home_form",
    "away_form",
    "rolling_point_diff"   # NEW
]

In [ ]:
team_strength.head()

In [ ]:
team_strength["rolling_point_diff"] = (
    (team_strength["rolling_point_diff"] - team_strength["rolling_point_diff"].mean()) /
    team_strength["rolling_point_diff"].std()
)
team_strength[["index", "rolling_point_diff"]].head()

below i am filtering based on current teams

In [ ]:
current_teams = [
    "Hawks","Celtics","Nets","Hornets","Bulls","Cavaliers","Mavericks",
    "Nuggets","Pistons","Warriors","Rockets","Pacers","Clippers",
    "Lakers","Grizzlies","Heat","Bucks","Timberwolves","Pelicans",
    "Knicks","Thunder","Magic","76ers","Suns","Trail Blazers",
    "Kings","Spurs","Raptors","Jazz","Wizards"
]

df_modern = df_modern[
    df_modern["hometeamName"].isin(current_teams) &
    df_modern["awayteamName"].isin(current_teams)
]

next is a win rate feature

In [ ]:
win_rate = df_modern.groupby("hometeamName")["HOME_WIN"].mean()

team_strength = team_strength.merge(
    win_rate.rename("win_rate"),
    left_on="index",
    right_index=True,
    how="left"
)

# Stage 6: Rebuild ML Model with new features

In [ ]:
ml_df = ml_df.merge(
    team_strength[["index", "rolling_point_diff", "win_rate"]],
    left_on="hometeamName",
    right_on="index",
    how="left"
).rename(columns={
    "rolling_point_diff": "home_rolling_point_diff",
    "win_rate": "home_win_rate"
}).drop(columns=["index"])

In [ ]:
ml_df = ml_df.merge(
    team_strength[["index", "rolling_point_diff", "win_rate"]],
    left_on="awayteamName",
    right_on="index",
    how="left"
).rename(columns={
    "rolling_point_diff": "away_rolling_point_diff",
    "win_rate": "away_win_rate"
}).drop(columns=["index"])

In [ ]:
features = [
    "home_strength",
    "away_strength",
    "home_form",
    "away_form",
    "home_rolling_point_diff",
    "away_rolling_point_diff",
    "home_win_rate",
    "away_win_rate"
]

In [ ]:
ml_df = ml_df.dropna(subset=features)

In [ ]:
X = ml_df[features]
y = ml_df["HOME_WIN"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

preds = model.predict(X_test)
ml_accuracy = accuracy_score(y_test, preds)

print(f"ML Model Accuracy: {ml_accuracy:.2%}")

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "weight": model.coef_[0]
})

importance["abs_weight"] = importance["weight"].abs()
importance = importance.sort_values("abs_weight", ascending=False)

importance[["feature", "weight"]].head(10)